In [ ]:
import scanpy as sc

adata = sc.read_h5ad("combined_integrated.h5ad")

adata

In [ ]:
batch_map = {
    "d01j": "J",
    "d05j": "J",
    "d10j": "J",
    "d25_2j": "J",
    "d25_1j": "J",
    "d10m": "M",
    "d24m": "M"
}

time_map = {
    "d01j": "d01",
    "d05j": "d05",
    "d10j": "d10",
    "d25_2j": "d25",
    "d25_1j": "d25",
    "d10m": "d10",
    "d24m": "d25"
}

adata.obs["batch"] = adata.obs["sample"].map(batch_map)
adata.obs["timepoint"] = adata.obs["sample"].map(time_map)
adata.obs["batch_time"] = adata.obs["batch"] + "_" + adata.obs["timepoint"]

In [ ]:
adata.obsm["X_umap"] = adata.obsm["UMAP.INTEGRATED"].values

In [ ]:
import scanpy as sc

sc.pl.umap(adata, color=["sample", "batch", "timepoint", "batch_time"])

In [ ]:
del adata.obsm["X_umap"] 
adata.obsm["X_umap"] = adata.obsm["UMAP.INTEGRATED"].values

In [ ]:
adata.obsm["X_umap"].shape

In [ ]:
adata.obs["sample"].value_counts()

In [ ]:
cluster_key = "integrated_clusters_0.1"
adata.obs["cluster"] = adata.obs[cluster_key].astype(str)
markers = [  "Lct","Maf","Aoc1","Neu1","Alpi","Anpep","Apoa4","Chga","Chgb",
  "Neurod1","Cpe","Muc2","Defa24","Agr2","Ccl6","Mki67","Krt8",
  "Msi1","Olfm4","Lgr5","Ascl2","Tert","Hopx","Gkn3","Dclk1",
  "Pou2f3","Trpm5","Avil"]
import scanpy as sc

sc.pl.dotplot(
    adata,
    var_names=markers,
    groupby="cluster",
    standard_scale="var"
)

In [ ]:
target_cluster = "0"   # change this to your cluster ID
adata.obs["cluster_group"] = adata.obs["cluster"].apply(
    lambda x: "target_cluster" if x == target_cluster else "other_clusters"
)
import pandas as pd

result = (
    adata.obs
    .groupby(["sample", "cluster_group"])["percent.mt"]
    .mean()
    .reset_index()
)
result.pivot(index="sample", columns="cluster_group", values="percent.mt")
count_table = (
    adata.obs
    .groupby(["sample", "cluster_group"])
    .size()
    .reset_index(name="n_cells")
)
summary = result.merge(count_table, on=["sample", "cluster_group"])
summary

In [ ]:
cluster_id = "0"   # change this

adata_sub = adata[adata.obs["cluster"] == "0"].copy()
gene_groups = {'Crypt': ['Rps8', 'Rpl13', 'Rps2', 'Rpl32', 'Ptma', 'Rps18', 'Rplp2', 'Rps19', 'Rps15a', 'Rpl23a'], 'V1': ['Gpx1', 'Plac8', 'Rpl10', 'Atp5g1', 'Fau', 'Lypd8', 'Rps29', 'Oat', 'Rps14', 'Rpl39'], 'V2': ['Fabp1', 'Aldh1a1', '2210407C18Rik', 'Ckmt1', 'Cox7a1', 'Gsta1', 'Fabp2', 'Maoa', 'Gstm3', 'Cat'], 'V3': ['Fabp1', 'Ace', 'Leap2', 'Slc43a2', 'Cyp4v3', 'Cat', 'Fam213a', 'Aldh1a1', 'Pdzk1', 'Cyp3a25'], 'V4': ['Cyp3a11', 'Lct', 'Sectm1b', 'Pdzk1', 'Asah2', 'Slc5a12', 'Slc7a15', 'Mxi1', 'Pck1', 'Scamp5'], 'V5': ['Apoa4', 'Apoc3', 'H2-K1', 'Sepp1', 'Clca4b', 'Enpp3', 'Slc6a8', 'Cyp3a13', 'Slc6a19', 'Slc34a2'], 'V6': ['Slc28a2', 'Clca4a', '2010109I03Rik', 'Ada', 'Nt5e', 'Apoa4', 'Cldn4', 'Muc3', 'Ifrd1', 'Serpinb1a']}
import scanpy as sc

for name, genes in gene_groups.items():
    genes = [g for g in genes if g in adata_sub.var_names]
    
    sc.tl.score_genes(
        adata_sub,
        gene_list=genes,
        score_name=name
    )
for b in adata_sub.obs["batch"].unique():
    
    sc.pl.umap(
        adata_sub[adata_sub.obs["batch"] == b],
        color=list(gene_groups.keys()),
        title=f"Batch: {b}",
        cmap="viridis"
    )


In [ ]:
import pandas as pd
import numpy as np

ref = pd.read_csv("table_D_zonation_reconstruction.tsv", sep="\t") # from Moor et al.
ref = ref.set_index("Gene name")

zones = ["Crypt", "V1", "V2", "V3", "V4", "V5", "V6"]

ref_profiles = ref[[f"{z}_mean" for z in zones]].copy()
ref_profiles.columns = zones
cluster_id = "0"
adata_sub = adata[adata.obs["cluster"] == cluster_id].copy()
shared_genes = list(set(adata_sub.var_names) & set(ref_profiles.index))

print(f"{len(shared_genes)} shared genes")
adata_sub2 = adata_sub[:, shared_genes].copy()
ref_profiles = ref_profiles.loc[shared_genes]
X = adata_sub2.X

if hasattr(X, "toarray"):
    X = X.toarray()
R = ref_profiles.values.T   # zones x genes
from scipy.stats import pearsonr

scores = np.zeros((X.shape[0], len(zones)))

for i in range(X.shape[0]):          # cells
    for j in range(len(zones)):      # zones
        scores[i, j] = pearsonr(X[i, :], R[j, :])[0]

best_idx = np.argmax(scores, axis=1)

adata_sub.obs["zone"] = [zones[i] for i in best_idx]
import scanpy as sc

sc.pl.umap(
    adata_sub,
    color=["zone", "zone_score"],
    palette="tab10"
)
pd.crosstab(adata_sub.obs["sample"], adata_sub.obs["zone"])


In [ ]:
import pandas as pd
import numpy as np

ref = pd.read_csv("table_D_zonation_reconstruction.tsv", sep="\t")
ref = ref.set_index("Gene name")

zones = ["Crypt", "V1", "V2", "V3", "V4", "V5", "V6"]

ref_profiles = ref[[f"{z}_mean" for z in zones]].copy()
ref_profiles.columns = zones
cluster_id = "0"
adata_sub = adata[adata.obs["cluster"] == cluster_id].copy()
shared_genes = list(set(adata_sub.var_names) & set(ref_profiles.index))

print(f"{len(shared_genes)} shared genes")
adata_sub2 = adata_sub[:, shared_genes].copy()
ref_profiles = ref_profiles.loc[shared_genes]
X = adata_sub2.X

if hasattr(X, "toarray"):
    X = X.toarray()
R = ref_profiles.values.T   # zones x genes
from scipy.stats import zscore

Xz = zscore(X, axis=1)
Rz = zscore(R, axis=1)

scores = Xz @ Rz.T / Xz.shape[1]

best_idx = np.argmax(scores, axis=1)

adata_sub.obs["zone"] = [zones[i] for i in best_idx]
import scanpy as sc

sc.pl.umap(
    adata_sub,
    color=["zone", "zone_score"],
    palette="tab10"
)
pd.crosstab(adata_sub.obs["sample"], adata_sub.obs["zone"])


In [ ]:
import pandas as pd
import numpy as np

ref = pd.read_csv("table_D_zonation_reconstruction.tsv", sep="\t")
ref = ref.set_index("Gene name")

mean_cols = [c for c in ref.columns if c.endswith("_mean")]
zones = [c.replace("_mean", "") for c in mean_cols]

ref_means = ref[mean_cols]
ref_means.columns = zones
shared = ref_means.index.intersection(adata_sub.var_names)

X = adata_sub[:, shared].X
if hasattr(X, "toarray"):
    X = X.toarray()

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import zscore

scores = cosine_similarity(
    zscore(X, axis=1),
    zscore(ref_means.loc[shared].T.values, axis=1)
)

adata_sub.obs["zone"] = np.array(zones)[scores.argmax(axis=1)]

sc.pl.umap(adata_sub, color="zone")

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc

from scipy.stats import zscore
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# PARAMETERS
# =========================================================

cluster_id = "0"          # cluster to analyze
MIN_EXPR = 0.001          # minimum reference expression
KEEP_VAR_QUANTILE = 0.5   # keep top variable reference genes
MIN_CONFIDENCE = 0.3      # optional confidence filter

# =========================================================
# SUBSET TARGET CLUSTER
# =========================================================

adata_sub = adata[adata.obs["cluster"] == cluster_id].copy()

print(f"Cells in cluster {cluster_id}: {adata_sub.n_obs}")

# =========================================================
# LOAD REFERENCE TABLE
# =========================================================

ref = pd.read_csv(
    "table_D_zonation_reconstruction.tsv",
    sep="\t"
)

ref = ref.set_index("Gene name")

# ---------------------------------------------------------
# Identify zones automatically
# ---------------------------------------------------------

mean_cols = [c for c in ref.columns if c.endswith("_mean")]

zones = [c.replace("_mean", "") for c in mean_cols]

ref_means = ref[mean_cols].copy()
ref_means.columns = zones

print("\nZones:")
print(zones)

# =========================================================
# FILTER INFORMATIVE REFERENCE GENES
# =========================================================

# expression strength
expr_strength = ref_means.max(axis=1)

# variability across zones
gene_var = ref_means.var(axis=1)

keep = (
    (expr_strength > MIN_EXPR) &
    (gene_var > gene_var.quantile(KEEP_VAR_QUANTILE))
)

ref_means = ref_means.loc[keep]

print(f"\nReference genes after filtering: {len(ref_means)}")

# =========================================================
# INTERSECT WITH ADATA GENES
# =========================================================

shared_genes = ref_means.index.intersection(adata_sub.var_names)

ref_means = ref_means.loc[shared_genes]

print(f"Shared genes with dataset: {len(shared_genes)}")

# =========================================================
# BUILD GENE WEIGHTS
# =========================================================

# specificity:
# high if enriched in one zone

specificity = (
    ref_means.max(axis=1) /
    (ref_means.mean(axis=1) + 1e-9)
)

# expression strength
expr_strength = ref_means.max(axis=1)

# final weights
weights = specificity * expr_strength

# normalize weights
weights = weights / weights.mean()

# =========================================================
# EXTRACT CELL EXPRESSION
# =========================================================

X = adata_sub[:, shared_genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

# =========================================================
# Z-SCORE NORMALIZATION
# =========================================================

# cells
Xz = zscore(X, axis=1, nan_policy="omit")

# reference zones
Rz = zscore(
    ref_means.T.values,
    axis=1,
    nan_policy="omit"
)

# replace NaNs
Xz = np.nan_to_num(Xz)
Rz = np.nan_to_num(Rz)

# =========================================================
# APPLY WEIGHTS
# =========================================================

w = weights.loc[shared_genes].values

Xw = Xz * w
Rw = Rz * w

# =========================================================
# COMPUTE SIMILARITY
# =========================================================

scores = cosine_similarity(Xw, Rw)

# =========================================================
# ASSIGN ZONES
# =========================================================

best_zone_idx = scores.argmax(axis=1)

adata_sub.obs["zone"] = np.array(zones)[best_zone_idx]

# confidence score
adata_sub.obs["zone_score"] = scores.max(axis=1)

# =========================================================
# OPTIONAL: FILTER LOW-CONFIDENCE CELLS
# =========================================================

adata_sub.obs["high_confidence"] = (
    adata_sub.obs["zone_score"] > MIN_CONFIDENCE
)

print("\nZone counts:")
print(adata_sub.obs["zone"].value_counts())

# =========================================================
# VISUALIZE UMAP
# =========================================================

sc.pl.umap(
    adata_sub,
    color=["zone", "zone_score"],
    cmap="viridis",
    frameon=False
)

# =========================================================
# CELL COUNTS PER SAMPLE + ZONE
# =========================================================

counts = (
    adata_sub.obs
    .groupby(["sample", "zone"])
    .size()
    .reset_index(name="n_cells")
)

print("\nCell counts per sample and zone:")
print(counts)

# =========================================================
# MITOCHONDRIAL PERCENTAGE
# =========================================================

mt_summary = (
    adata_sub.obs
    .groupby(["sample", "zone"])["percent.mt"]
    .agg(["mean", "median", "std"])
    .reset_index()
)

print("\nMitochondrial statistics:")
print(mt_summary)

# =========================================================
# COMBINE COUNTS + MITO
# =========================================================

summary = counts.merge(
    mt_summary,
    on=["sample", "zone"]
)

print("\nCombined summary:")
print(summary)

# =========================================================
# SAVE OUTPUTS
# =========================================================

summary.to_csv(
    "zone_sample_mito_summary.tsv",
    sep="\t",
    index=False
)

adata_sub.obs.to_csv(
    "cell_zone_assignments.tsv",
    sep="\t"
)

# =========================================================
# OPTIONAL HEATMAP
# =========================================================

import seaborn as sns
import matplotlib.pyplot as plt

heatmap_df = summary.pivot(
    index="sample",
    columns="zone",
    values="mean"
)

plt.figure(figsize=(8, 5))

sns.heatmap(
    heatmap_df,
    cmap="viridis",
    annot=True
)

plt.title("Mean mitochondrial %")
plt.tight_layout()
plt.show()

# =========================================================
# OPTIONAL STACKED COMPOSITION PLOT
# =========================================================

composition = pd.crosstab(
    adata_sub.obs["sample"],
    adata_sub.obs["zone"],
    normalize="index"
)

composition.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5)
)

plt.ylabel("Fraction of cells")
plt.title("Zone composition by sample")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import zscore

# =====================================================
# LOG TRANSFORM (critical for RNA-seq)
# =====================================================

ref_log = np.log1p(ref_means)
X = adata_sub[:, shared_genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

X = np.log1p(X)

# =====================================================
# Z-NORMALIZE (per gene)
# =====================================================

Xz = zscore(X, axis=0)
Xz = np.nan_to_num(Xz)

Rz = zscore(ref_log.T.values, axis=1)
Rz = np.nan_to_num(Rz)

# =====================================================
# GENE WEIGHTS (robust + non-collapse version)
# =====================================================

# signal-to-noise across zones
gene_mean = ref_log.mean(axis=1)
gene_std = ref_log.std(axis=1) 

# how specific a gene is to ANY zone
specificity = ref_log.max(axis=1) / (gene_mean )

weights = specificity / gene_std

weights = weights / np.mean(weights)

w = weights.loc[shared_genes].values

# =====================================================
# WEIGHTED DOT PRODUCT MODEL (IMPORTANT CHANGE)
# =====================================================

# each zone is a vector in gene space
# each cell is projected onto each zone vector

scores = Xz @ (Rz.T)

# apply gene weights in a stable way
scores = scores * np.mean(w)

# =====================================================
# SOFTMAX (prevents collapse)
# =====================================================

scores = scores - scores.max(axis=1, keepdims=True)

probs = np.exp(scores)
probs = probs / probs.sum(axis=1, keepdims=True)

# =====================================================
# ASSIGNMENT
# =====================================================

adata_sub.obs["zone"] = np.array(zones)[np.argmax(probs, axis=1)]

adata_sub.obs["zone_prob"] = np.max(probs, axis=1)

# entropy = uncertainty (VERY IMPORTANT)
adata_sub.obs["zone_entropy"] = -np.sum(probs * np.log(probs), axis=1)#
print(adata_sub.obs["zone"].value_counts())

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# GLOBAL UMAP LIMITS
# (same scale for every plot)
# =========================================================

global_umap = adata_sub.obsm["X_umap"]

offset=1
xmin = global_umap[:, 0].min()-offset
xmax = global_umap[:, 0].max()+offset

ymin = global_umap[:, 1].min()-offset
ymax = global_umap[:, 1].max()+offset

# =========================================================
# MAIN UMAP (ALL CELLS)
# =========================================================

sc.pl.umap(
    adata_sub,
    color=["zone", "zone_prob", "zone_entropy"],
    cmap="viridis",
    frameon=False
)

# =========================================================
# UMAPS PER SAMPLE
# =========================================================

for s in adata_sub.obs["sample"].unique():

    adata_s = adata_sub[adata_sub.obs["sample"] == s]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # -----------------------------
    # zone
    # -----------------------------

    sc.pl.umap(
        adata_s,
        color="zone",
        frameon=False,
        ax=axes[0],
        show=False,
        title=f"{s} - zone"
    )

    axes[0].set_xlim(xmin, xmax)
    axes[0].set_ylim(ymin, ymax)

    # -----------------------------
    # zone_prob
    # -----------------------------

    sc.pl.umap(
        adata_s,
        color="zone_prob",
        cmap="viridis",
        frameon=False,
        ax=axes[1],
        show=False,
        title=f"{s} - zone_prob"
    )

    axes[1].set_xlim(xmin, xmax)
    axes[1].set_ylim(ymin, ymax)

    plt.tight_layout()
    plt.show()

# =========================================================
# UMAPS PER BATCH
# =========================================================

for b in adata_sub.obs["batch"].unique():

    adata_b = adata_sub[adata_sub.obs["batch"] == b]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # -----------------------------
    # zone
    # -----------------------------

    sc.pl.umap(
        adata_b,
        color="zone",
        frameon=False,
        ax=axes[0],
        show=False,
        title=f"{b} - zone"
    )

    axes[0].set_xlim(xmin, xmax)
    axes[0].set_ylim(ymin, ymax)

    # -----------------------------
    # zone_prob
    # -----------------------------

    sc.pl.umap(
        adata_b,
        color="zone_prob",
        cmap="viridis",
        frameon=False,
        ax=axes[1],
        show=False,
        title=f"{b} - zone_prob"
    )

    axes[1].set_xlim(xmin, xmax)
    axes[1].set_ylim(ymin, ymax)

    plt.tight_layout()
    plt.show()

# =========================================================
# FILTER FOR COMPOSITION
# =========================================================

comp = adata_sub.obs.copy()
comp = comp[comp["zone_prob"] > 0.3]

# =========================================================
# ZONE COMPOSITION
# =========================================================

composition = pd.crosstab(
    comp["sample"],
    comp["zone"],
    normalize="index"
)

composition = composition.sort_index()

# =========================================================
# USE SAME COLORS AS UMAP
# =========================================================

# Scanpy stores categorical colors here
zone_order = adata_sub.obs["zone"].cat.categories.tolist()
zone_colors = adata_sub.uns["zone_colors"]

# map zone -> color
color_dict = dict(zip(zone_order, zone_colors))

# colors in plotting order
plot_colors = [color_dict[z] for z in composition.columns]

# =========================================================
# STACKED BARPLOT
# =========================================================

ax = composition.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    color=plot_colors
)

plt.ylabel("Fraction of cells")
plt.title("Zone composition by sample")

# =========================================================
# LEGEND OUTSIDE + REVERSED ONLY IN LEGEND
# =========================================================

handles, labels = ax.get_legend_handles_labels()

plt.legend(
    handles[::-1],
    labels[::-1],
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    title="Zone"
)

plt.tight_layout()
plt.savefig(
    "zone_composition_by_sample.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "zone_composition_by_sample.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

In [ ]:
# =========================================================
# TOTAL CELL COUNTS PER ZONE
# =========================================================

counts = pd.crosstab(
    comp["sample"],
    comp["zone"]
)

counts = counts.sort_index()

# =========================================================
# SAME COLORS AS UMAP
# =========================================================

zone_order = adata_sub.obs["zone"].cat.categories.tolist()
zone_colors = adata_sub.uns["zone_colors"]

color_dict = dict(zip(zone_order, zone_colors))

plot_colors = [color_dict[z] for z in counts.columns]

# =========================================================
# STACKED BARPLOT (TOTAL CELLS)
# =========================================================

ax = counts.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    color=plot_colors
)

plt.ylabel("Number of cells")
plt.title("Total cells per zone by sample")

# =========================================================
# LEGEND OUTSIDE + REVERSED ORDER
# =========================================================

handles, labels = ax.get_legend_handles_labels()

plt.legend(
    handles[::-1],
    labels[::-1],
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    title="Zone"
)

plt.tight_layout()
plt.savefig(
    "total_cells_per_zone_by_sample.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "total_cells_per_zone_by_sample.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

In [ ]:
# =========================================================
# CUSTOM MULTI-PANEL SAMPLE LAYOUT
# =========================================================

fig, axes = plt.subplots(
    5,
    2,
    figsize=(12, 25)
)

axes = np.array(axes)

# turn off all axes first
for ax_row in axes:
    for ax in ax_row:
        ax.axis("off")

# =========================================================
# SAMPLE POSITIONS
# =========================================================

sample_layout = {
    "d01j":   (0, 0),
    "d05j":   (1, 0),
    "d10j":   (2, 0),
    "d10m":   (2, 1),
    "d25_1j": (3, 0),
    "d24m":   (3, 1),
    "d25_2j": (4, 0),
}

# =========================================================
# STORE LEGEND INFO
# =========================================================

legend_handles = None
legend_labels = None

# =========================================================
# PLOT EACH SAMPLE
# =========================================================

for sample, (r, c) in sample_layout.items():

    if sample not in adata_sub.obs["sample"].unique():
        print(f"Skipping missing sample: {sample}")
        continue

    adata_s = adata_sub[
        adata_sub.obs["sample"] == sample
    ]

    ax = axes[r, c]
    ax.axis("on")

    sc.pl.umap(
        adata_s,
        color="zone",
        frameon=False,
        ax=ax,
        show=False,
        title=sample,
        legend_loc="right margin"   # needed to generate handles
    )

    # SAME SCALE
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # capture legend once
    if legend_handles is None:
        legend_handles, legend_labels = ax.get_legend_handles_labels()

    # remove subplot legend
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

# =========================================================
# GLOBAL LEGEND
# =========================================================

fig.legend(
    legend_handles[::-1],
    legend_labels[::-1],
    loc="center right",
    bbox_to_anchor=(1.02, 0.5),
    title="Zone"
)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.savefig(
    "abs_ent_umap_by_sample.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "abs_ent_umap_by_sample.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. ZONES
# =========================================================

foreground_zones = ["V4", "V5", "V6"]
background_zones = ["V1", "V2", "V3"]

# =========================================================
# 2. GROUPING
# =========================================================

adata_sub.obs["de_group"] = "other"

mask_j = (
    adata_sub.obs["sample"].isin(["d25_1j", "d25_2j"]) &
    adata_sub.obs["zone"].isin(foreground_zones + background_zones)
)

mask_m = (
    (adata_sub.obs["sample"] == "d24m") &
    adata_sub.obs["zone"].isin(foreground_zones + background_zones)
)

adata_sub.obs.loc[mask_j, "de_group"] = "J"
adata_sub.obs.loc[mask_m, "de_group"] = "M"

# =========================================================
# 3. J DE (FG vs BG)
# =========================================================

adata_j = adata_sub[adata_sub.obs["de_group"] == "J"].copy()

adata_j.obs["zone_group"] = np.where(
    adata_j.obs["zone"].isin(foreground_zones),
    "fg",
    "bg"
)

sc.tl.rank_genes_groups(
    adata_j,
    groupby="zone_group",
    groups=["fg"],
    reference="bg",
    method="wilcoxon"
)

df_j = sc.get.rank_genes_groups_df(adata_j, group="fg")[
    ["names", "logfoldchanges", "pvals_adj"]
].rename(columns={
    "logfoldchanges": "logFC_j",
    "pvals_adj": "padj_j"
})

# =========================================================
# 4. M DE (FG vs BG)
# =========================================================

adata_m = adata_sub[adata_sub.obs["de_group"] == "M"].copy()

adata_m.obs["zone_group"] = np.where(
    adata_m.obs["zone"].isin(foreground_zones),
    "fg",
    "bg"
)

sc.tl.rank_genes_groups(
    adata_m,
    groupby="zone_group",
    groups=["fg"],
    reference="bg",
    method="wilcoxon"
)

df_m = sc.get.rank_genes_groups_df(adata_m, group="fg")[
    ["names", "logfoldchanges", "pvals_adj"]
].rename(columns={
    "logfoldchanges": "logFC_m",
    "pvals_adj": "padj_m"
})

# =========================================================
# 5. MERGE
# =========================================================

df = pd.merge(df_j, df_m, on="names", how="outer")

df_sig = df[
    (df["padj_j"] < 0.05) |
    (df["padj_m"] < 0.05)
].copy()

df_sig["logFC_j"] = df_sig["logFC_j"].fillna(0)
df_sig["logFC_m"] = df_sig["logFC_m"].fillna(0)

# =========================================================
# 6. TRANSFORM
# =========================================================

def transform(x):
    return np.sign(x) * np.log10(1 + np.abs(x))

df_sig["x"] = transform(df_sig["logFC_m"])
df_sig["y"] = transform(df_sig["logFC_j"])

# =========================================================
# 7. SIGNIFICANCE CLASS (COLOR ONLY)
# =========================================================

def sig_class(r):
    j = r["padj_j"] < 0.05
    m = r["padj_m"] < 0.05

    if j and m:
        return "both"
    elif m:
        return "m_only"
    else:
        return "j_only"

df_sig["sig"] = df_sig.apply(sig_class, axis=1)

# =========================================================
# 8. PLOT BASE
# =========================================================

plt.figure(figsize=(8, 8))

colors = {
    "both": "black",
    "m_only": "orange",
    "j_only": "purple"
}

for cat, sub in df_sig.groupby("sig"):
    plt.scatter(
        sub["x"],
        sub["y"],
        s=20,
        alpha=0.7,
        c=colors[cat],
        label=cat
    )

plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)

plt.xlabel("M (d24m) logFC (log10-compressed)")
plt.ylabel("J (d25_1j + d25_2j) logFC (log10-compressed)")
plt.title("DE comparison across conditions")

# =========================================================
# 9. QUADRANTS
# =========================================================

q1 = (df_sig["x"] > 0) & (df_sig["y"] > 0)
q2 = (df_sig["x"] < 0) & (df_sig["y"] > 0)
q3 = (df_sig["x"] < 0) & (df_sig["y"] < 0)
q4 = (df_sig["x"] > 0) & (df_sig["y"] < 0)

# =========================================================
# 10. QUADRANT COUNTS (BY COLOR CLASS)
# =========================================================

def count(mask, cls):
    return np.sum(mask & (df_sig["sig"] == cls))

def qstats(mask):
    return (
        count(mask, "both"),
        count(mask, "m_only"),
        count(mask, "j_only")
    )

q1_b, q1_m, q1_j = qstats(q1)
q2_b, q2_m, q2_j = qstats(q2)
q3_b, q3_m, q3_j = qstats(q3)
q4_b, q4_m, q4_j = qstats(q4)

# =========================================================
# 11. TEXT BOXES
# =========================================================

xlim = plt.xlim()
ylim = plt.ylim()

x_left  = xlim[0] + 0.15 * (xlim[1] - xlim[0])
x_right = xlim[1] - 0.15 * (xlim[1] - xlim[0])

y_top   = ylim[1] - 0.15 * (ylim[1] - ylim[0])
y_bot   = ylim[0] + 0.15 * (ylim[1] - ylim[0])

def annotate(xp, yp, b, m, j):
    plt.text(
        xp, yp,
        f"both: {b}\nj_only: {j}\nm_only: {m}",
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(facecolor="white", alpha=0.6)
    )

annotate(x_right, y_top, q1_b, q1_m, q1_j)
annotate(x_left,  y_top, q2_b, q2_m, q2_j)
annotate(x_left,  y_bot, q3_b, q3_m, q3_j)
annotate(x_right, y_bot, q4_b, q4_m, q4_j)

# =========================================================
# 12. TOP 10 LABELS PER QUADRANT
# (color = SIGNIFICANCE, not quadrant)
# =========================================================

df_sig["score"] = np.abs(df_sig["x"]) + np.abs(df_sig["y"])

def label(mask):
    sub = df_sig[mask].nlargest(10, "score")

    for _, r in sub.iterrows():
        plt.text(
            r["x"],
            r["y"],
            r["names"],
            fontsize=7,
            color=colors[r["sig"]],
            alpha=0.9
        )

label(q1)
label(q2)
label(q3)
label(q4)

# =========================================================
# LEGEND
# =========================================================

plt.legend(title="Significance")
plt.tight_layout()
plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. ZONES
# =========================================================

foreground_zones = ["V4", "V5", "V6"]
background_zones = ["V1", "V2", "V3"]

# =========================================================
# 2. GROUP ASSIGNMENT
# =========================================================

adata_sub.obs["de_group"] = "other"

mask_j = (
    adata_sub.obs["sample"].isin(["d25_1j", "d25_2j"]) &
    adata_sub.obs["zone"].isin(foreground_zones + background_zones)
)

mask_m = (
    (adata_sub.obs["sample"] == "d24m") &
    adata_sub.obs["zone"].isin(foreground_zones + background_zones)
)

adata_sub.obs.loc[mask_j, "de_group"] = "J"
adata_sub.obs.loc[mask_m, "de_group"] = "M"

# =========================================================
# 3. J DE (FG vs BG)
# =========================================================

adata_j = adata_sub[adata_sub.obs["de_group"] == "J"].copy()

adata_j.obs["zone_group"] = np.where(
    adata_j.obs["zone"].isin(foreground_zones),
    "fg",
    "bg"
)

sc.tl.rank_genes_groups(
    adata_j,
    groupby="zone_group",
    groups=["fg"],
    reference="bg",
    method="wilcoxon"
)

df_j = sc.get.rank_genes_groups_df(adata_j, group="fg")[[
    "names", "logfoldchanges", "pvals_adj"
]].rename(columns={
    "logfoldchanges": "logFC_j",
    "pvals_adj": "padj_j"
})

# =========================================================
# 4. M DE (FG vs BG)
# =========================================================

adata_m = adata_sub[adata_sub.obs["de_group"] == "M"].copy()

adata_m.obs["zone_group"] = np.where(
    adata_m.obs["zone"].isin(foreground_zones),
    "fg",
    "bg"
)

sc.tl.rank_genes_groups(
    adata_m,
    groupby="zone_group",
    groups=["fg"],
    reference="bg",
    method="wilcoxon"
)

df_m = sc.get.rank_genes_groups_df(adata_m, group="fg")[[
    "names", "logfoldchanges", "pvals_adj"
]].rename(columns={
    "logfoldchanges": "logFC_m",
    "pvals_adj": "padj_m"
})

# =========================================================
# 5. MERGE
# =========================================================

df = pd.merge(df_j, df_m, on="names", how="outer")

df["logFC_j"] = df["logFC_j"].fillna(0)
df["logFC_m"] = df["logFC_m"].fillna(0)

df = df[(df["padj_j"] < 0.05) | (df["padj_m"] < 0.05)].copy()

# =========================================================
# 6. SIGNIFICANCE CLASS
# =========================================================

def sig_class(r):
    j = r["padj_j"] < 0.05
    m = r["padj_m"] < 0.05
    if j and m:
        return "both"
    elif m:
        return "m_only"
    else:
        return "j_only"

df["sig"] = df.apply(sig_class, axis=1)

colors = {
    "both": "black",
    "m_only": "orange",
    "j_only": "purple"
}

# =========================================================
# 7. TRANSFORM
# =========================================================

def double_log(x):
    return np.sign(x) * np.log10(1 + np.abs(x))

df["x"] = df["logFC_m"] #double_log(df["logFC_m"])
df["y"] = df["logFC_j"] #double_log(df["logFC_j"])

# =========================================================
# 8. AGREEMENT METRIC
# =========================================================

eps = 1e-9
df["agreement"] = (2 * df["x"] * df["y"]) / (df["x"]**2 + df["y"]**2 + eps)

# =========================================================
# 9. SCATTER PLOT
# =========================================================

plt.figure(figsize=(7, 7))

for cat, sub in df.groupby("sig"):
    plt.scatter(sub["x"], sub["y"], s=18, alpha=0.7,
                c=colors[cat], label=cat)

plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)

plt.xlabel("M (d24m) log2FC (log10-compressed)")
plt.ylabel("J (d25) log2FC (log10-compressed)")
plt.title("DE comparison (union of significant genes)")

# =========================================================
# 10. QUADRANTS + COUNTS
# =========================================================

q1 = (df["x"] > 0) & (df["y"] > 0)
q2 = (df["x"] < 0) & (df["y"] > 0)
q3 = (df["x"] < 0) & (df["y"] < 0)
q4 = (df["x"] > 0) & (df["y"] < 0)

def count(mask, cls):
    return np.sum(mask & (df["sig"] == cls))

def qstats(mask):
    return (
        count(mask, "both"),
        count(mask, "m_only"),
        count(mask, "j_only")
    )

q1_b, q1_m, q1_j = qstats(q1)
q2_b, q2_m, q2_j = qstats(q2)
q3_b, q3_m, q3_j = qstats(q3)
q4_b, q4_m, q4_j = qstats(q4)

xlim = plt.xlim()
ylim = plt.ylim()

x_left  = xlim[0] + 0.15 * (xlim[1] - xlim[0])
x_right = xlim[1] - 0.15 * (xlim[1] - xlim[0])
y_top   = ylim[1] - 0.15 * (ylim[1] - ylim[0])
y_bot   = ylim[0] + 0.15 * (ylim[1] - ylim[0])

def annotate(xp, yp, b, m, j):
    plt.text(xp, yp,
             f"both: {b}\nj_only: {j}\nm_only: {m}",
             ha="center", va="center", fontsize=10,
             bbox=dict(facecolor="white", alpha=0.6))

annotate(x_right, y_top, q1_b, q1_m, q1_j)
annotate(x_left,  y_top, q2_b, q2_m, q2_j)
annotate(x_left,  y_bot, q3_b, q3_m, q3_j)
annotate(x_right, y_bot, q4_b, q4_m, q4_j)

# =========================================================
# 11. TOP GENES PER QUADRANT
# =========================================================

df["score"] = np.abs(df["x"]) + np.abs(df["y"])

def label(mask):
    sub = df[mask].nlargest(10, "score")
    for _, r in sub.iterrows():
        plt.text(r["x"], r["y"], r["names"],
                 fontsize=7, color=colors[r["sig"]])

label(q1)
label(q2)
label(q3)
label(q4)

plt.legend(title="Significance")
plt.tight_layout()
plt.savefig(
    "DE_robustness_MvsJ.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "DE_robustness_MvsJ.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

# =========================================================
# 12. AGREEMENT DISTRIBUTION
# =========================================================

plt.figure(figsize=(7, 4))

bin_edges = np.arange(-1, 1.05, 0.05)

sns.histplot(df["agreement"], bins=bin_edges, kde=False)

median_val = df["agreement"].median()

plt.axvline(median_val, color="red", linestyle="--", linewidth=2)

# shifted median label
plt.text(median_val - 0.08,
         plt.ylim()[1] * 0.85,
         f"median = {median_val:.2f}",
         color="red",
         rotation=90,
         va="top")

plt.xlabel("Gene agreement [-1, 1]")
plt.title("Concordance distribution")

plt.tight_layout()
plt.savefig(
    "robustness_score_MvsJ.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "robustness_score_MvsJ.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

# =========================================================
# 13. SUMMARY
# =========================================================

print("Genes:", len(df))
print("Mean agreement:", df["agreement"].mean())
print("Median agreement:", median_val)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. ZONES
# =========================================================

foreground_zones = ["V4", "V5", "V6"]
background_zones = ["V1", "V2", "V3"]

# =========================================================
# 2. SUBSET J ONLY (two replicates)
# =========================================================

adata = adata_sub[
    adata_sub.obs["sample"].isin(["d25_1j", "d25_2j"])
].copy()

# =========================================================
# 3. SPLIT INTO TWO REPLICATES
# =========================================================

adata_r1 = adata[adata.obs["sample"] == "d25_1j"].copy()
adata_r2 = adata[adata.obs["sample"] == "d25_2j"].copy()

def run_de(adata_in):
    adata_in.obs["zone_group"] = np.where(
        adata_in.obs["zone"].isin(foreground_zones),
        "fg",
        "bg"
    )

    sc.tl.rank_genes_groups(
        adata_in,
        groupby="zone_group",
        groups=["fg"],
        reference="bg",
        method="wilcoxon"
    )

    df = sc.get.rank_genes_groups_df(adata_in, group="fg")[
        ["names", "logfoldchanges", "pvals_adj"]
    ].rename(columns={
        "logfoldchanges": "logFC",
        "pvals_adj": "padj"
    })

    return df

df_r1 = run_de(adata_r1).rename(columns={
    "logFC": "logFC_r1",
    "padj": "padj_r1"
})

df_r2 = run_de(adata_r2).rename(columns={
    "logFC": "logFC_r2",
    "padj": "padj_r2"
})

# =========================================================
# 4. MERGE (KEEP ALL GENES, BUT DO NOT IMPUTE ZEROS)
# =========================================================

df = pd.merge(df_r1, df_r2, on="names", how="outer")

df = df[(df["padj_r1"] < 0.05) | (df["padj_r2"] < 0.05)].copy()

# =========================================================
# 5. SIGNIFICANCE CLASS
# =========================================================

def sig_class(r):
    r1 = r["padj_r1"] < 0.05
    r2 = r["padj_r2"] < 0.05

    if r1 and r2:
        return "both"
    elif r1:
        return "r1_only"
    else:
        return "r2_only"

df["sig"] = df.apply(sig_class, axis=1)

colors = {
    "both": "black",
    "r1_only": "orange",
    "r2_only": "purple"
}

# =========================================================
# 6. REMOVE MISSING BEFORE TRANSFORM (CRITICAL FIX)
# =========================================================

df = df.dropna(subset=["logFC_r1", "logFC_r2"]).copy()

# =========================================================
# 7. DOUBLE LOG TRANSFORM
# =========================================================

def double_log(x):
    return np.sign(x) * np.log10(1 + np.abs(x))

df["x"] = df["logFC_r1"] #double_log(df["logFC_r1"])
df["y"] = df["logFC_r2"] #double_log(df["logFC_r2"])

# =========================================================
# 8. AGREEMENT METRIC
# =========================================================

eps = 1e-9
df["agreement"] = (2 * df["x"] * df["y"]) / (df["x"]**2 + df["y"]**2 + eps)

# =========================================================
# 9. SCATTER PLOT
# =========================================================

plt.figure(figsize=(7, 7))

for cat, sub in df.groupby("sig"):
    plt.scatter(
        sub["x"],
        sub["y"],
        s=18,
        alpha=0.7,
        c=colors[cat],
        label=cat
    )

plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)

plt.xlabel("d25_1j log2FC (log10-compressed)")
plt.ylabel("d25_2j log2FC (log10-compressed)")
plt.title("Replicate consistency (V4–V6 vs V1–V3)")

# =========================================================
# 10. QUADRANT ANNOTATIONS
# =========================================================

q1 = (df["x"] > 0) & (df["y"] > 0)
q2 = (df["x"] < 0) & (df["y"] > 0)
q3 = (df["x"] < 0) & (df["y"] < 0)
q4 = (df["x"] > 0) & (df["y"] < 0)

def count(mask, cls):
    return np.sum(mask & (df["sig"] == cls))

def qstats(mask):
    return (
        count(mask, "both"),
        count(mask, "r1_only"),
        count(mask, "r2_only")
    )

q1_b, q1_r1, q1_r2 = qstats(q1)
q2_b, q2_r1, q2_r2 = qstats(q2)
q3_b, q3_r1, q3_r2 = qstats(q3)
q4_b, q4_r1, q4_r2 = qstats(q4)

xlim = plt.xlim()
ylim = plt.ylim()

x_left  = xlim[0] + 0.15 * (xlim[1] - xlim[0])
x_right = xlim[1] - 0.15 * (xlim[1] - xlim[0])

y_top   = ylim[1] - 0.15 * (ylim[1] - ylim[0])
y_bot   = ylim[0] + 0.15 * (ylim[1] - ylim[0])

def annotate(xp, yp, b, r1, r2):
    plt.text(
        xp, yp,
        f"both: {b}\nr1_only: {r1}\nr2_only: {r2}",
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(facecolor="white", alpha=0.6)
    )

annotate(x_right, y_top, q1_b, q1_r1, q1_r2)
annotate(x_left,  y_top, q2_b, q2_r1, q2_r2)
annotate(x_left,  y_bot, q3_b, q3_r1, q3_r2)
annotate(x_right, y_bot, q4_b, q4_r1, q4_r2)

# =========================================================
# 11. TOP GENES PER QUADRANT (FIXED COLORING)
# =========================================================

df["score"] = np.abs(df["x"]) + np.abs(df["y"])

def label(mask):
    sub = df[mask].nlargest(10, "score")
    for _, r in sub.iterrows():
        plt.text(
            r["x"],
            r["y"],
            r["names"],
            fontsize=7,
            color=colors[r["sig"]],   # ✅ FIXED HERE
            alpha=0.9
        )

label(q1)
label(q2)
label(q3)
label(q4)

plt.legend(title="Replicate significance")
plt.tight_layout()
plt.savefig(
    "DE_robustness_JvsJ.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "DE_robustness_JvsJ.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

# =========================================================
# 12. AGREEMENT DISTRIBUTION
# =========================================================

plt.figure(figsize=(7, 4))

bin_edges = np.arange(-1, 1.05, 0.05)

sns.histplot(df["agreement"], bins=bin_edges, kde=False)

median_val = df["agreement"].median()

plt.axvline(median_val, color="red", linestyle="--")

plt.text(
    median_val - 0.08,
    plt.ylim()[1] * 0.85,
    f"median = {median_val:.2f}",
    color="red",
    rotation=90,
    va="top"
)

plt.xlabel("Gene agreement [-1, 1]")
plt.title("Concordance distribution")

plt.tight_layout()
plt.savefig(
    "robustness_score_JvsJ.svg",
    format="svg",
    bbox_inches="tight"
)
plt.savefig(
    "robustness_score_JvsJ.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()

# =========================================================
# 13. SUMMARY
# =========================================================

print("Genes:", len(df))
print("Median agreement:", median_val)
print("Mean agreement:", df["agreement"].mean())

In [ ]:
df[(df["x"] == 0) & (df["y"] == 0)][["names", "logFC_r1", "logFC_r2", "padj_r1", "padj_r2"]].head(20)

In [ ]:
import gseapy as gp

# =========================================================
# GO ENRICHMENT
# =========================================================

enr = gp.enrichr(
    gene_list=top_genes,
    gene_sets=[
        "GO_Biological_Process_2023",
        "GO_Molecular_Function_2023",
        "GO_Cellular_Component_2023"
    ],
    organism="mouse",
    outdir="go_results",
)

# results table
go_results = enr.results

# show top terms
print(
    go_results[
        ["Gene_set", "Term", "Adjusted P-value"]
    ].head(20)
)
gp.barplot(
    go_results,
    column="Adjusted P-value",
    group="Gene_set",
    top_term=10,
    figsize=(8, 6)
)
ax = gp.barplot(
    go_results,
    column="Adjusted P-value",
    group="Gene_set",
    top_term=10,
    figsize=(8, 6)
)

plt.savefig(
    "GO_barplot_d24m_V5V6.pdf",
    format="pdf",
    bbox_inches="tight"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

umap = adata_sub.obsm["X_umap"]
obs = adata_sub.obs.copy()

# =========================================================
# BIN CONTINUOUS PSEUDOTIME INTO TRAJECTORY STEPS
# =========================================================

n_bins = 50  # controls smoothness

adata_sub.obs["pt_bin"] = pd.cut(
    adata_sub.obs["pseudotime"],
    bins=n_bins,
    labels=False
)

# =========================================================
# PER SAMPLE TRAJECTORY (ROBUST VERSION)
# =========================================================

def plot_sample_trajectory(sample):

    ad = adata_sub[adata_sub.obs["sample"] == sample]
    um = ad.obsm["X_umap"]
    od = ad.obs.copy()

    if len(ad) < 50:
        print(f"Skipping {sample}: too few cells")
        return

    # -----------------------------------------------------
    # compute mean UMAP position per pseudotime bin
    # -----------------------------------------------------
    traj = (
        od.assign(umap1=um[:, 0], umap2=um[:, 1])
        .groupby("pt_bin")[["umap1", "umap2"]]
        .mean()
        .dropna()
        .sort_index()
    )

    coords = traj.values

    if len(coords) < 5:
        print(f"Skipping {sample}: unstable trajectory")
        return

    # -----------------------------------------------------
    # plot
    # -----------------------------------------------------
    plt.figure(figsize=(6, 5))

    sns.scatterplot(
        x=um[:, 0],
        y=um[:, 1],
        hue=od["zone"],
        palette="tab20",
        s=5,
        alpha=0.5,
        linewidth=0
    )

    # trajectory line
    plt.plot(
        coords[:, 0],
        coords[:, 1],
        color="black",
        linewidth=3,
        zorder=10
    )

    plt.title(f"Trajectory (pseudotime) — {sample}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# =========================================================
# RUN
# =========================================================

for s in adata_sub.obs["sample"].unique():
    plot_sample_trajectory(s)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# =====================================================
# BUILD MATRICES
# =====================================================

count_pivot = summary.pivot(
    index="sample",
    columns="zone",
    values="n_cells"
).fillna(0)

mt_pivot = summary.pivot(
    index="sample",
    columns="zone",
    values="mean"
).fillna(0)

mt_pivot = mt_pivot.reindex_like(count_pivot)

# =====================================================
# LOG TRANSFORM (SAFE)
# =====================================================

log_counts = np.log1p(count_pivot)

# mask true zeros (so they become black)
mask_zero = count_pivot == 0

# =====================================================
# COLORMAP (BLACK FOR ZERO)
# =====================================================

base_cmap = sns.color_palette("mako", as_cmap=True)

cmap = base_cmap.copy()
cmap.set_bad(color="black")

# apply mask
log_counts_masked = log_counts.mask(mask_zero)

# =====================================================
# ANNOTATION = MITO %
# =====================================================

annot = mt_pivot.round(2).astype(str)
annot[annot == "0.0"] = ""

# =====================================================
# PLOT
# =====================================================

plt.figure(figsize=(11, 6))

sns.heatmap(
    log_counts_masked,
    cmap=cmap,
    annot=annot,
    fmt="",
    linewidths=0.5,
    cbar_kws={
        "label": "log(1 + cell count)"
    }
)

plt.title(
    "Zone distribution per sample\n"
    "(color = log cell counts, black = 0, annotation = mito %)"
)

plt.xlabel("Zone")
plt.ylabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# CLEAN DATA
# =========================================================

plot_df = adata_sub.obs[["zone", "percent.mt", "sample"]].copy()
plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna()

zone_order = sorted(plot_df["zone"].unique())
sample_order = sorted(plot_df["sample"].unique())

plot_df["zone"] = pd.Categorical(plot_df["zone"], categories=zone_order, ordered=True)
plot_df["sample"] = pd.Categorical(plot_df["sample"], categories=sample_order, ordered=True)

# =========================================================
# MEDIANS (zone × sample)
# =========================================================

medians = (
    plot_df.groupby(["zone", "sample"])["percent.mt"]
    .median()
    .reset_index()
)

# map categorical positions
zone_pos = {z: i for i, z in enumerate(zone_order)}
sample_pos = {s: i for i, s in enumerate(sample_order)}

# spacing for dodge (must match seaborn behavior roughly)
dodge_width = 0.8
n_samples = len(sample_order)

# =========================================================
# PLOT
# =========================================================

plt.figure(figsize=(14, 6))

ax = sns.stripplot(
    data=plot_df,
    x="zone",
    y="percent.mt",
    hue="sample",
    dodge=True,
    jitter=0.25,
    alpha=0.5,
    size=3,
    zorder=2   # dots ABOVE lines
)

# =========================================================
# MEDIAN LINES (correct alignment + behind dots)
# =========================================================

for _, row in medians.iterrows():
    z = row["zone"]
    s = row["sample"]
    m = row["percent.mt"]

    if pd.isna(m):
        continue

    base_x = zone_pos[z]

    # compute consistent dodge offset
    offset = (sample_pos[s] - (n_samples - 1) / 2) * (dodge_width / n_samples)

    x = base_x + offset

    ax.hlines(
        y=m,
        xmin=x - 0.12,
        xmax=x + 0.12,
        color="black",
        linewidth=2,
        zorder=100   # IMPORTANT: behind points
    )

# =========================================================
# FORMATTING
# =========================================================

plt.xlabel("Zone")
plt.ylabel("Mitochondrial %")
plt.title("Mitochondrial % per zone × sample (median lines correctly aligned)")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    title="Sample"
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import zscore
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================
# PARAMETERS
# =====================================================

cluster_id = "2"

TOP_GENES_PER_ZONE = 30
MIN_EXPR = 0.001
EPS = 1e-9

# =====================================================
# SUBSET TARGET CLUSTER
# =====================================================

adata_sub = adata[
    adata.obs["cluster"] == cluster_id
].copy()

print(f"\nCells in cluster {cluster_id}: {adata_sub.n_obs}")

# =====================================================
# LOAD REFERENCE
# =====================================================

ref = pd.read_csv(
    "table_D_zonation_reconstruction.tsv",
    sep="\t"
)

ref = ref.set_index("Gene name")

# =====================================================
# IDENTIFY ZONES
# =====================================================

zones = [
    c.replace("_mean", "")
    for c in ref.columns
    if c.endswith("_mean")
]

print("\nZones:")
print(zones)

# =====================================================
# BUILD REFERENCE MATRIX
# =====================================================

ref_means = ref[[f"{z}_mean" for z in zones]].copy()
ref_means.columns = zones

# remove weak genes
ref_means = ref_means[
    ref_means.max(axis=1) > MIN_EXPR
]

# =====================================================
# FIND STRONG ZONE MARKERS
# =====================================================

selected_genes = []

for zone in zones:

    target = ref_means[zone]

    others = ref_means.drop(columns=zone).mean(axis=1)

    # zone enrichment
    log2fc = np.log2(
        (target + EPS) /
        (others + EPS)
    )

    # specificity
    specificity = target / (
        ref_means.mean(axis=1) + EPS
    )

    # combined score
    score = log2fc * specificity * target

    score = (
        score
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    top = (
        score.sort_values(ascending=False)
        .head(TOP_GENES_PER_ZONE)
    )

    print(f"\n=== {zone} top markers ===")

    for gene, val in top.items():
        print(f"{gene:20s} {val:.3f}")

    selected_genes.extend(top.index.tolist())

# unique genes
selected_genes = list(set(selected_genes))

# =====================================================
# INTERSECT WITH DATASET
# =====================================================

shared = list(
    set(selected_genes)
    .intersection(adata_sub.var_names)
)

print(f"\nSelected informative genes: {len(shared)}")

# =====================================================
# BUILD REFERENCE CENTROIDS
# =====================================================

R = ref_means.loc[shared]

# genes x zones
R = R[zones]

# log transform
R = np.log1p(R)

# z-score PER GENE
Rz = zscore(R, axis=1)

Rz = np.nan_to_num(Rz)

# transpose:
# zones x genes
Rz = Rz.T

# =====================================================
# BUILD CELL MATRIX
# =====================================================

X = adata_sub[:, shared].X

if hasattr(X, "toarray"):
    X = X.toarray()

X = np.log1p(X)

# z-score PER CELL
Xz = zscore(X, axis=1)

Xz = np.nan_to_num(Xz)

# =====================================================
# COMPUTE SIMILARITIES
# =====================================================

scores = cosine_similarity(
    Xz,
    Rz
)

# =====================================================
# ASSIGN ZONES
# =====================================================

best_idx = scores.argmax(axis=1)

adata_sub.obs["zone"] = np.array(zones)[best_idx]

# best similarity
adata_sub.obs["zone_score"] = scores.max(axis=1)

# confidence margin
sorted_scores = np.sort(scores, axis=1)

adata_sub.obs["zone_margin"] = (
    sorted_scores[:, -1] -
    sorted_scores[:, -2]
)

# =====================================================
# PRINT ZONE COUNTS
# =====================================================

print("\nZone counts:")
print(
    adata_sub.obs["zone"]
    .value_counts()
)

# =====================================================
# STORE INDIVIDUAL ZONE SCORES
# =====================================================

score_df = pd.DataFrame(
    scores,
    columns=zones,
    index=adata_sub.obs_names
)

for z in zones:
    adata_sub.obs[f"{z}_score"] = score_df[z]

# =====================================================
# UMAP VISUALIZATION
# =====================================================

sc.pl.umap(
    adata_sub,
    color=[
        "zone",
        "zone_score",
        "zone_margin"
    ],
    frameon=False,
    cmap="viridis"
)

# =====================================================
# SHOW INDIVIDUAL ZONE SIGNALS
# =====================================================

sc.pl.umap(
    adata_sub,
    color=[f"{z}_score" for z in zones],
    ncols=3,
    cmap="viridis",
    frameon=False
)

# =====================================================
# SAMPLE × ZONE SUMMARY
# =====================================================

summary = (
    adata_sub.obs
    .groupby(["sample", "zone"])["percent.mt"]
    .agg(
        n_cells="size",
        mean_mt="mean",
        median_mt="median"
    )
    .reset_index()
)

print("\nSummary:")
print(summary)

# =====================================================
# SAVE OUTPUTS
# =====================================================

summary.to_csv(
    "zone_sample_summary.tsv",
    sep="\t",
    index=False
)

adata_sub.obs.to_csv(
    "cell_zone_assignments.tsv",
    sep="\t"
)

# =====================================================
# HEATMAP:
# color = mitochondrial %
# text = cell counts
# =====================================================

mt_pivot = summary.pivot(
    index="sample",
    columns="zone",
    values="mean_mt"
)

count_pivot = summary.pivot(
    index="sample",
    columns="zone",
    values="n_cells"
)

# remove empty rows/cols
mt_pivot = mt_pivot.dropna(how="all", axis=0)
mt_pivot = mt_pivot.dropna(how="all", axis=1)

count_pivot = count_pivot.loc[
    mt_pivot.index,
    mt_pivot.columns
]

# annotations
annot = count_pivot.copy()

annot = annot.fillna(0)

annot = annot.round(0).astype(int).astype(str)

annot[annot == "0"] = ""

# plot
plt.figure(figsize=(10, 6))

sns.heatmap(
    mt_pivot,
    annot=annot,
    fmt="",
    cmap="viridis",
    linewidths=0.5,
    cbar_kws={
        "label": "Mean mitochondrial %"
    }
)

plt.title(
    "Zone distribution per sample\n"
    "(numbers = cell counts, color = mito %)"
)

plt.ylabel("Sample")
plt.xlabel("Zone")

plt.tight_layout()
plt.show()
# annotations:
# show counts only if > 0
# =====================================================
# CLEAN ANNOTATIONS
# =====================================================

annot = count_pivot.copy()

# replace NaN with 0 temporarily
annot = annot.fillna(0)

# convert to strings
annot = annot.round(0).astype(int).astype(str)

# hide zeros
annot[annot == "0"] = ""

# plot
plt.figure(figsize=(10, 6))

sns.heatmap(
    mt_pivot,
    annot=annot,
    fmt="",
    cmap="viridis",
    linewidths=0.5,
    cbar_kws={
        "label": "Mean mitochondrial %"
    }
)

plt.title(
    "Zone distribution per sample\n"
    "(numbers = cell counts, color = mito %)"
)

plt.ylabel("Sample")
plt.xlabel("Zone")

plt.tight_layout()
plt.show()

# =====================================================
# ZONE COMPOSITION PER SAMPLE
# =====================================================

composition = pd.crosstab(
    adata_sub.obs["sample"],
    adata_sub.obs["zone"],
    normalize="index"
)

composition.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5)
)

plt.ylabel("Fraction")
plt.title("Zone composition")
plt.tight_layout()
plt.show()

In [ ]:
ref_means

In [ ]:
counts = (
    adata_sub.obs
    .groupby(["sample", "zone"])
    .size()
    .reset_index(name="n_cells")
)
mt = (
    adata_sub.obs
    .groupby(["sample", "zone"])["percent.mt"]
    .mean()
    .reset_index(name="mean_percent_mt")
)
summary = counts.merge(mt, on=["sample", "zone"])
summary

In [ ]:

import scanpy as sc
import anndata
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import igraph as ig
import scanpy.external as sce
import harmonypy as hm
import scvi
#import louvain

# Print library versions
for lib in [sc, anndata, np, pd, matplotlib, sns, ig]:
    print(f"{lib.__name__}: {lib.__version__}")

# Configure scanpy settings for optimal performance
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')
np.random.seed(0)

In [ ]:

samples = {
    "d10j": ("./d10j/", "J", "d10"),
    "d25_2j": ("./d25_2j/", "J", "d25"),
    "d25_1j": ("./d25_1j/", "J", "d25"),
    "d10m": ("./d10m/", "M", "d10"),
    "d24m": ("./d24m/", "M", "d25")
}

adatas = []

for sample, (path, batch, timepoint) in samples.items():
    ad = sc.read_10x_mtx(path, var_names="gene_symbols", cache=True)
    ad.var_names_make_unique()

    ad.obs["sample"] = sample
    ad.obs["batch"] = batch
    ad.obs["timepoint"] = timepoint

    adatas.append(ad)

adata_good = sc.concat(adatas, join="inner")
 

In [ ]:
adata = adata_good.copy()

In [ ]:
# =========================================================
# QC
# =========================================================
adata.var["mt"] = adata.var_names.str.startswith("mt-")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True
)

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs["pct_counts_mt"] < 20].copy()

# store raw counts
adata.layers["counts"] = adata.X.copy()

# =========================================================
# NORMALIZATION
# =========================================================
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# =========================================================
# HVGs (batch-aware)
# =========================================================
sc.pp.highly_variable_genes(
    adata,
    batch_key="batch",
    flavor="seurat_v3",
    n_top_genes=3000
)

adata = adata[:, adata.var.highly_variable].copy()

# =========================================================
# SCALE + PCA
# =========================================================
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")

assert adata.obsm["X_pca"].shape[1] == 50

# =========================================================
# HARMONY BATCH CORRECTION
# =========================================================
ho = hm.run_harmony(
    adata.obsm["X_pca"],
    adata.obs,
    vars_use=["batch"],
    theta=4,
    nclust=50
)

adata.obsm["X_pca_harmony"] = ho.Z_corr

# =========================================================
# NEIGHBORS + UMAP (on corrected space)
# =========================================================
sc.pp.neighbors(
    adata,
    use_rep="X_pca_harmony",
    n_neighbors=15
)

sc.tl.umap(adata)

# =========================================================
# DIAGNOSTIC ANNOTATION
# =========================================================
adata.obs["batch_time"] = (
    adata.obs["batch"].astype(str)
    + "_"
    + adata.obs["timepoint"].astype(str)
)

# =========================================================
# VISUALIZATION
# =========================================================
sc.pl.umap(adata, color=["batch"])
sc.pl.umap(adata, color=["timepoint"])
sc.pl.umap(adata, color=["sample"])
sc.pl.umap(adata, color=["batch_time"])

# =========================================================
# OPTIONAL: CLUSTERING
# =========================================================
sc.tl.leiden(adata, resolution=0.5)
sc.pl.umap(adata, color="leiden")

In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
torch.cuda.empty_cache()
torch.set_float32_matmul_precision("high")
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

In [ ]:
adata = adata_good.copy()
# =========================================================
# SPEED BOOST (GPU Tensor Cores)
# =========================================================
torch.set_float32_matmul_precision("high")

# =========================================================
# QC (optimized + correct thresholds)
# =========================================================

adata.obs_names_make_unique()

adata.var["mt"] = adata.var_names.str.startswith("mt-")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True
)

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# ⚠️ 100% mito filter is too permissive (does nothing useful)
adata = adata[adata.obs["pct_counts_mt"] < 100].copy()

# =========================================================
# STORE RAW COUNTS (critical for scVI)
# =========================================================
adata.layers["counts"] = adata.X.copy()

# =========================================================
# scVI SETUP (FIXED - no dataloader_kwargs here)
# =========================================================
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch"
)

# =========================================================
# MODEL (optimized architecture for speed + stability)
# =========================================================
model = scvi.model.SCVI(
    adata,
    n_latent=10, # 20   # ↓ faster, still enough for structure
    n_layers=1, #2
    n_hidden=64 # 64
)

# =========================================================
# TRAINING (MAIN SPEED OPTIMIZATION)
# =========================================================
model.train(
    max_epochs=10,       # ↓ 200 → 100 (usually enough)
    batch_size=1024, # 512      # ↑ GPU utilization
    accelerator="gpu",
    devices=1,
    precision=32   # 🔥 huge VRAM reduction
)

# =========================================================
# LATENT SPACE
# =========================================================
adata.obsm["X_scVI"] = model.get_latent_representation()

# =========================================================
# GRAPH + UMAP
# =========================================================
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
sc.tl.umap(adata)

# =========================================================
# VISUALIZATION
# =========================================================
sc.pl.umap(adata, color=["batch"])
sc.pl.umap(adata, color=["timepoint"])
sc.pl.umap(adata, color=["sample"])

# =========================================================
# CLUSTERING
# =========================================================
sc.tl.leiden(adata, resolution=0.5)
sc.pl.umap(adata, color="leiden")

In [ ]:
model.save("scvi_model/", overwrite=True) # scvi_model/
adata.write("adata_scvi.h5ad") # adata_scvi.h5ad

In [ ]:
import scanpy as sc
import scanpy.external as sce
import torch
import os

# =========================================================
# OPTIONAL GPU / TORCH SETTINGS (safe to keep)
# =========================================================
torch.set_float32_matmul_precision("high")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# =========================================================
# COPY DATA
# =========================================================
adata = adata_good.copy()

# =========================================================
# QC
# =========================================================
adata.obs_names_make_unique()

adata.var["mt"] = adata.var_names.str.startswith("mt-")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

# (optional: make this stricter if needed)
adata = adata[adata.obs["pct_counts_mt"] < 60].copy()

# =========================================================
# NORMALIZATION (important for Scanorama)
# =========================================================
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# =========================================================
# HIGHLY VARIABLE GENES
# =========================================================
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    batch_key="batch"
)

adata = adata[:, adata.var["highly_variable"]].copy()

# =========================================================
# SCALE + PCA (orig.reduction = "pca")
# =========================================================
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50)

# =========================================================
# SCANORAMA INTEGRATION (CCA-like)
# =========================================================
sce.pp.scanorama_integrate(
    adata,
    key="batch",              # batch column in adata.obs
    basis="X_pca",            # input = PCA
    adjusted_basis="X_scanorama"
)

# =========================================================
# NEIGHBORS + UMAP (use integrated space)
# =========================================================
sc.pp.neighbors(adata, use_rep="X_scanorama", n_neighbors=15)
sc.tl.umap(adata)

# =========================================================
# CLUSTERING
# =========================================================
sc.tl.leiden(adata, resolution=0.5)

# =========================================================
# VISUALIZATION
# =========================================================
sc.pl.umap(adata, color=["batch"])
sc.pl.umap(adata, color=["leiden"])
sc.pl.umap(adata, color=["timepoint"])
sc.pl.umap(adata, color=["sample"])

In [ ]:
sce.pp.scanorama_integrate(
    adata,
    key="batch",              # batch column in adata.obs
    basis="X_pca",            # input = PCA
    adjusted_basis="X_scanorama"
)

# =========================================================
# NEIGHBORS + UMAP (use integrated space)
# =========================================================
sc.pp.neighbors(adata, use_rep="X_scanorama", n_neighbors=15)
sc.tl.umap(adata)

# =========================================================
# CLUSTERING
# =========================================================
sc.tl.leiden(adata, resolution=0.5)

# =========================================================
# VISUALIZATION
# =========================================================
sc.pl.umap(adata, color=["batch"])
sc.pl.umap(adata, color=["leiden"])
sc.pl.umap(adata, color=["timepoint"])
sc.pl.umap(adata, color=["sample"])

In [ ]:
import scanpy as sc
import harmonypy as hm
# =========================================================
adata = adata_good.copy()
# -------------------------
# 1. QC METRICS (MOUSE)
# -------------------------
# Mouse mitochondrial genes start with "mt-"
adata.var["mt"] = adata.var_names.str.startswith("mt-")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True
)

# -------------------------
# 2. BASIC FILTERING
# -------------------------
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

# Optional (recommended sanity filtering)
adata = adata[adata.obs["pct_counts_mt"] < 60, :].copy()

# -------------------------
# 3. STORE RAW COUNTS
# -------------------------
adata.layers["counts"] = adata.X.copy()

# -------------------------
# 4. NORMALIZATION
# -------------------------
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# -------------------------
# 5. HVGs (IMPROVED)
# -------------------------
sc.pp.highly_variable_genes(
    adata,
    batch_key="batch",
    flavor="seurat_v3",
    n_top_genes=4000,
    subset=True
)
####
adata.var["highly_variable"] = (
    adata.var["highly_variable"] & ~adata.var["mt"]
)

# subset to HVGs
adata = adata[:, adata.var["highly_variable"]].copy()
####


# -------------------------
# 6. SCALE + PCA
# -------------------------
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50)

# sanity check
assert adata.obsm["X_pca"].shape == (adata.n_obs, 50)

# -------------------------
# 7. HARMONY (CORRECT MODERN USAGE)
# -------------------------
ho = hm.run_harmony(
    adata.obsm["X_pca"],
    adata.obs,
    vars_use=["batch"],
    theta=6,      # slightly stronger correction
    nclust=50
)

adata.obsm["X_pca_harmony"] = ho.Z_corr

# -------------------------
# 8. NEIGHBORS + UMAP
# -------------------------
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)

# -------------------------
# 9. VISUALIZATION
# -------------------------
sc.pl.umap(adata, color="batch")
sc.pl.umap(adata, color="timepoint")
sc.pl.umap(adata, color="sample")

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

In [ ]:
sc.pl.umap(adata, color="batch")
sc.pl.umap(adata, color="timepoint")
sc.pl.umap(adata, color="sample")

adata_sub = adata[~adata.obs["sample"].isin([ "d10j"])].copy()
sc.pl.umap(adata_sub, color="sample")

adata_sub = adata[~adata.obs["sample"].isin(["d25_1j", "d25_2j"])].copy()
sc.pl.umap(adata_sub, color="sample")

adata_sub = adata[~adata.obs["sample"].isin(["d10m"])].copy()
sc.pl.umap(adata_sub, color="sample")

adata_sub = adata[~adata.obs["sample"].isin(["d24m"])].copy()
sc.pl.umap(adata_sub, color="sample")




#sc.pl.umap(adata, color="leiden")

In [ ]:
# Print summary of the AnnData object
print(adata)

# Number of cells (observations)
print(f"Number of cells: {adata.n_obs}")

# Number of genes (variables)
print(f"Number of genes: {adata.n_vars}")

# Preview first few cell (obs) and gene (var) names
print(f"First 5 cell IDs: {adata.obs_names[:5].tolist()}")
print(f"First 5 gene names: {adata.var_names[:5].tolist()}")

# Matrix sparsity info
nonzero = (adata.X != 0).sum()
total = adata.shape[0] * adata.shape[1]
print(f"Data sparsity: {(1 - nonzero / total):.2%}")

In [ ]:
# Annotate mitochondrial, ribosomal, and hemoglobin genes
adata.var['mt'] = adata.var_names.str.startswith('mt-')      # Mitochondrial genes
adata.var['ribo'] = adata.var_names.str.startswith(('Rps', 'Rpl'))  # Ribosomal genes
adata.var['hb'] = adata.var_names.str.startswith('Hbb')       # Hemoglobin genes

print(f"Mitochondrial genes: {adata.var['mt'].sum()}")
print(f"Ribosomal genes: {adata.var['ribo'].sum()}")
print(f"Hemoglobin genes: {adata.var['hb'].sum()}")

# Compute QC metrics using these annotations
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt', 'ribo', 'hb'],
    percent_top=None,
    log1p=False,
    inplace=True
)

# Visualize QC metrics
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=ax[0], show=False)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=ax[1], show=False)
plt.tight_layout(); plt.show()


print("\nQC metrics calculated:")
print(f"QC columns in adata.obs: {[col for col in adata.obs.columns if 'total' in col or 'pct' in col or 'n_genes' in col]}")

# Basic statistics
print(f"\nBasic statistics:")
print(f"Mean genes per cell: {adata.obs['n_genes_by_counts'].mean():.2f}")
print(f"Mean UMIs per cell: {adata.obs['total_counts'].mean():.2f}")
print(f"Mean mitochondrial percentage: {adata.obs['pct_counts_mt'].mean():.2f}%")
if 'pct_counts_ribo' in adata.obs.columns:
    print(f"Mean ribosomal percentage: {adata.obs['pct_counts_ribo'].mean():.2f}%")

In [ ]:
# Print initial statistics
print(f"Before filtering:")
print(f"  Cells: {adata.n_obs}")
print(f"  Genes: {adata.n_vars}")

# Basic gene filter: keep genes expressed in at least 3 cells
sc.pp.filter_genes(adata, min_cells=3)

# Cell-level QC filtering
adata = adata[
    (adata.obs['n_genes_by_counts'] > 500) & # Remove cells with too few genes (likely empty droplets)
    (adata.obs['n_genes_by_counts'] < 5000) & # Remove cells with too many genes (potential doublets)
    (adata.obs['pct_counts_mt'] < 100) & # Remove cells with high mitochondrial content (stressed/dying)
    (adata.obs['total_counts'] > 1000) & # Remove low-UMI cells (low-quality)
    (adata.obs['total_counts'] < 30000), 
    :
]

# Optional stricter PBMC-style filters
# adata = adata[adata.obs['n_genes_by_counts'] < 2500, :]
# adata = adata[adata.obs['pct_counts_mt'] < 10, :]

# Final cleanup (redundant with earlier gene filtering, but safe to repeat)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=500)

# Calculate gene statistics
adata.var['n_cells'] = (adata.X > 0).sum(axis=0).A1
adata.var['mean_counts'] = adata.X.mean(axis=0).A1
adata.var['pct_dropout'] = 100 * (1 - adata.var['n_cells'] / adata.n_obs)

print(f"\nGene statistics:")
print(f"  Mean cells per gene: {adata.var['n_cells'].mean():.2f}")
print(f"  Mean counts per gene: {adata.var['mean_counts'].mean():.2f}")
print(f"  Mean dropout percentage: {adata.var['pct_dropout'].mean():.2f}%")

print(f"\nFinal filtered data:")
print(f"  Cells: {adata.n_obs}")
print(f"  Genes: {adata.n_vars}")

In [ ]:
# Create a figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Quality control metrics', fontsize=16, y=0.98)

# Plot 1: Number of genes per cell (violin plot)
axes[0,0].violinplot([adata.obs['n_genes_by_counts']], positions=[0], showmeans=True, showmedians=True)
axes[0,0].set_ylabel('Number of genes')
axes[0,0].set_title('Genes per cell')
axes[0,0].set_xticks([0])
axes[0,0].set_xticklabels(['Cells'])

# Plot 2: Total UMI counts per cell (violin plot)
axes[0,1].violinplot([adata.obs['total_counts']], positions=[0], showmeans=True, showmedians=True)
axes[0,1].set_ylabel('Total UMI counts')
axes[0,1].set_title('UMI counts per cell')
axes[0,1].set_xticks([0])
axes[0,1].set_xticklabels(['Cells'])

# Plot 3: Mitochondrial gene percentage (violin plot)
axes[0,2].violinplot([adata.obs['pct_counts_mt']], positions=[0], showmeans=True, showmedians=True)
axes[0,2].set_ylabel('Mitochondrial gene %')
axes[0,2].set_title('Mitochondrial gene %')
axes[0,2].set_xticks([0])
axes[0,2].set_xticklabels(['Cells'])

# Plot 4: Scatter plot - genes vs UMIs
axes[1,0].scatter(adata.obs['total_counts'], adata.obs['n_genes_by_counts'], alpha=0.6, s=1)
axes[1,0].set_xlabel('Total UMI counts')
axes[1,0].set_ylabel('Number of genes')
axes[1,0].set_title('Genes vs UMI counts')

# Plot 5: Scatter plot - UMIs vs mitochondrial %
axes[1,1].scatter(adata.obs['total_counts'], adata.obs['pct_counts_mt'], alpha=0.6, s=1)
axes[1,1].set_xlabel('Total UMI counts')
axes[1,1].set_ylabel('Mitochondrial gene %')
axes[1,1].set_title('UMI counts vs Mitochondrial %')

# Plot 6: Ribosomal gene percentage (violin plot)
axes[1,2].violinplot([adata.obs['pct_counts_ribo']], positions=[0], showmeans=True, showmedians=True)
axes[1,2].set_ylabel('Ribosomal gene %')
axes[1,2].set_title('Ribosomal gene %')
axes[1,2].set_xticks([0])
axes[1,2].set_xticklabels(['Cells'])

#plt.tight_layout()
#qc_plot_path = os.path.join(output_dir, 'qc_metrics.png')
#plt.savefig(qc_plot_path, dpi=300, bbox_inches='tight')
#plt.close()

#print(f"QC metrics plot saved at: {qc_plot_path}")

# Summary statistics
print("\nDetailed QC statistics:")
print("=" * 50)
for metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo']:
    if metric in adata.obs.columns:
        data = adata.obs[metric]
        print(f"{metric}:")
        print(f"  Mean: {data.mean():.2f}")
        print(f"  Median: {data.median():.2f}")
        print(f"  Std: {data.std():.2f}")
        print(f"  Min: {data.min():.2f}")
        print(f"  Max: {data.max():.2f}")
        print()

In [ ]:
# Normalize and log-transform the data
sc.pp.normalize_total(adata, target_sum=1e4) # Normalize to 10,000 counts per cell
sc.pp.log1p(adata) # Log-transform

# Identify highly variable genes (HVGs) for downstream analysis
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Visualize HVGs
plt.figure(figsize=(8, 4))
sc.pl.highly_variable_genes(adata, show=False)
plt.title('Highly variable genes')

# Subset the data to keep only highly variable genes
adata = adata[:, adata.var.highly_variable]

# Regress out unwanted sources of variation (technical effects)
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale each gene to unit variance and zero mean; clip extreme values
sc.pp.scale(adata, max_value=10)

In [ ]:
# Perform PCA on highly variable genes only
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

# Create PCA plots manually
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: PCA variance ratio
variance_ratio = adata.uns['pca']['variance_ratio']
axes[0,0].plot(range(1, len(variance_ratio)+1), variance_ratio, 'bo-', markersize=3)
axes[0,0].set_xlabel('Principal component')
axes[0,0].set_ylabel('Variance ratio')
axes[0,0].set_title('PCA variance ratio')
axes[0,0].set_yscale('log')

# Plot 2-4: PCA scatter plots
pca_coords = adata.obsm['X_pca']

# Color by total counts
scatter1 = axes[0,1].scatter(pca_coords[:, 0], pca_coords[:, 1], 
                            c=adata.obs['total_counts'], s=1, alpha=0.7, cmap='viridis')
axes[0,1].set_xlabel('PC1')
axes[0,1].set_ylabel('PC2')
axes[0,1].set_title('PCA - Total counts')
plt.colorbar(scatter1, ax=axes[0,1])

# Color by number of genes
scatter2 = axes[1,0].scatter(pca_coords[:, 0], pca_coords[:, 1], 
                            c=adata.obs['n_genes_by_counts'], s=1, alpha=0.7, cmap='viridis')
axes[1,0].set_xlabel('PC1')
axes[1,0].set_ylabel('PC2')
axes[1,0].set_title('PCA - Number of genes')
plt.colorbar(scatter2, ax=axes[1,0])

# Color by mitochondrial percentage
scatter3 = axes[1,1].scatter(pca_coords[:, 0], pca_coords[:, 1], 
                            c=adata.obs['pct_counts_mt'], s=1, alpha=0.7, cmap='viridis')
axes[1,1].set_xlabel('PC1')
axes[1,1].set_ylabel('PC2')
axes[1,1].set_title('PCA - Mitochondrial %')
plt.colorbar(scatter3, ax=axes[1,1])

plt.tight_layout()

# Print PCA statistics
print(f"PCA completed:")
print(f"  Number of principal components: {adata.obsm['X_pca'].shape[1]}")
print(f"  Variance explained by first PC: {adata.uns['pca']['variance_ratio'][0]:.3f}")
print(f"  Variance explained by first 10 PCs: {adata.uns['pca']['variance_ratio'][:10].sum():.3f}")
print(f"  Variance explained by first 50 PCs: {adata.uns['pca']['variance_ratio'][:50].sum():.3f}")

# Show top contributing genes for first few PCs
n_top_genes = 5
for pc in range(3):  # First 3 PCs
    pc_loadings = adata.varm['PCs'][:, pc]
    top_genes_idx = np.argsort(np.abs(pc_loadings))[-n_top_genes:]
    top_genes = adata.var_names[top_genes_idx]
    loadings_values = pc_loadings[top_genes_idx]
    print(f"\nTop {n_top_genes} genes for PC{pc+1}:")
    for gene, loading in zip(top_genes, loadings_values):
        print(f"  {gene}: {loading:.3f}")

# Visualizes cells in PCA space, colored by the expression level of the gene CST3 for example
#sc.pl.pca(adata, color='CST3', vmin=0, vmax=10)

In [ ]:
# Compute the neighborhood graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
# Compute UMAP
sc.tl.umap(adata, random_state=42)

# Run Leiden and Louvain clustering with igraph backend
sc.tl.leiden(
    adata,
    resolution=0.15,
    random_state=42,
    flavor="igraph",     # Use igraph instead of leidenalg
    directed=False,       # Required for igraph backend
    n_iterations=2,      # Matches upcoming default
    key_added='leiden_0.15'
)

# Run with different reolutions
sc.tl.leiden(adata, resolution=0.5, key_added='leiden_0.5')
sc.tl.leiden(adata, resolution=0.8, key_added='leiden_0.8')
sc.tl.leiden(adata, resolution=1.0, key_added='leiden_1.0')

sc.tl.louvain(
    adata,
    flavor="igraph",
    directed=False,
    random_state=42
)

for resolution in ['leiden_0.15', 'leiden_0.5', 'leiden_0.8', 'leiden_1.0']:
    n_clusters = len(adata.obs[resolution].unique())
    print(f"  {resolution}: {n_clusters} clusters")

# Set default clustering
adata.obs['clusters'] = adata.obs['leiden_0.15']

# Check cluster sizes
print(adata.obs['leiden_0.15'].value_counts().sort_index())
print(adata.obs['louvain'].value_counts().sort_index())

In [ ]:
# Create UMAP plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('UMAP visualization', fontsize=16)

# Plot 1: UMAP colored by clusters
umap_coords = adata.obsm['X_umap']
clusters = adata.obs['clusters'].astype(str)
unique_clusters = sorted(clusters.unique(), key=lambda x: int(x))

# Create a color map for clusters
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters)))
cluster_colors = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

for cluster in unique_clusters:
    mask = clusters == cluster
    axes[0,0].scatter(umap_coords[mask, 0], umap_coords[mask, 1], 
                     c=[cluster_colors[cluster]], label=f'Cluster {cluster}', s=1, alpha=0.7)
axes[0,0].set_xlabel('UMAP1')
axes[0,0].set_ylabel('UMAP2')
axes[0,0].set_title('Clusters (Leiden 0.15)')
axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=5)

# Plot 2: UMAP colored by total counts
scatter1 = axes[0,1].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['total_counts'], s=1, alpha=0.7, cmap='viridis')
axes[0,1].set_xlabel('UMAP1')
axes[0,1].set_ylabel('UMAP2')
axes[0,1].set_title('Total counts')
plt.colorbar(scatter1, ax=axes[0,1])

# Plot 3: UMAP colored by number of genes
scatter2 = axes[0,2].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['n_genes_by_counts'], s=1, alpha=0.7, cmap='viridis')
axes[0,2].set_xlabel('UMAP1')
axes[0,2].set_ylabel('UMAP2')
axes[0,2].set_title('Number of genes')
plt.colorbar(scatter2, ax=axes[0,2])

# Plot 4: UMAP colored by mitochondrial percentage
scatter3 = axes[1,0].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['pct_counts_mt'], s=1, alpha=0.7, cmap='viridis')
axes[1,0].set_xlabel('UMAP1')
axes[1,0].set_ylabel('UMAP2')
axes[1,0].set_title('Mitochondrial %')
plt.colorbar(scatter3, ax=axes[1,0])

# Plot 5: UMAP colored by ribosomal percentage
scatter4 = axes[1,1].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['pct_counts_ribo'], s=1, alpha=0.7, cmap='viridis')
axes[1,1].set_xlabel('UMAP1')
axes[1,1].set_ylabel('UMAP2')
axes[1,1].set_title('Ribosomal %')
plt.colorbar(scatter4, ax=axes[1,1])

# Plot 6: Different clustering resolution
clusters_05 = adata.obs['leiden_0.5'].astype(str)
unique_clusters_05 = sorted(clusters_05.unique(), key=lambda x: int(x))
colors_05 = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters_05)))
cluster_colors_05 = {cluster: colors_05[i] for i, cluster in enumerate(unique_clusters_05)}

for cluster in unique_clusters_05:
    mask = clusters_05 == cluster
    axes[1,2].scatter(umap_coords[mask, 0], umap_coords[mask, 1], 
                     c=[cluster_colors_05[cluster]], s=1, alpha=0.7)
axes[1,2].set_xlabel('UMAP1')
axes[1,2].set_ylabel('UMAP2')
axes[1,2].set_title('Clusters (Leiden 0.5)')

plt.tight_layout()
print(f"UMAP coordinates shape: {adata.obsm['X_umap'].shape}")

In [ ]:
sc.pl.umap(adata, color=['leiden_0.15', 'louvain'], legend_loc='on data')


In [ ]:
nt5e

bottom of villi: reg3a muc17 lypd8 or 6

In [ ]:
# Find marker genes for each cluster
sc.tl.rank_genes_groups(
    adata, 
    groupby='louvain', #'leiden_0.15'
    method='wilcoxon',
    key_added='rank_genes_groups',
    pts=True
)

sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False) # n_genes=20

# Optional: save results to DataFrame
#marker_df = sc.get.rank_genes_groups_df(adata, group='0')  # change group as needed

In [ ]:
# Define PBMC marker genes
pbmc_markers = {
    'bottom of villi': ['Reg3a', 'Lypd6', 'Lypd8'],
    'top of villi': ['Nt5e']

}


# Visualize expression to validate
sc.pl.dotplot(adata, var_names=pbmc_markers, groupby='leiden_0.15', use_raw=True)
sc.pl.matrixplot(adata,
    var_names=pbmc_markers,
    groupby='leiden_0.15',
    use_raw=True,
    standard_scale='var',  # Normalize gene expression across all clusters
    cmap='Reds'
)

In [ ]:
# Map clusters to cell types
cluster_annotation = {
    '0': 'CD4+ T cells',
    '1': 'Cytotoxic T/NK Cells', 
    '2': 'Dendritic Cells',
    '3': 'Monocytes',
    '4': 'FCGR3A+ Monocytes',
    '5': 'NK Cells',
    '6': 'B Cells',
    '7': 'CD8+ T cells',
    '8': 'Megakaryocytes',
    '9': 'Progenitor-like',
}

adata.obs['cell_type'] = adata.obs['leiden_0.15'].map(cluster_annotation)

# Plot annotated UMAP
sc.pl.umap(adata, color='cell_type', legend_loc='on data', 
           legend_fontsize=10, legend_fontoutline=2, frameon=False)

In [ ]:
sc.pl.umap(
    adata,
    #color=['Reg3a', 'Lypd6', 'Lypd8','Nt5e', 'Apobec1', 'Apob', 'Apoa4', 'Npc1l1', 'Slc28a2', 'Enpp3', 'Ada'],
    color=['Mki67', 'Stmn1', 'Tubb5', 'Ube2c', 'Mybl2'],
    use_raw=True,
    vmin=0,
    vmax=100,
    cmap='Reds',
    size=20
)